# Student Assignment Delay Predictor — Complete ML Project

This notebook predicts whether an assignment is likely to be late (`1`) or on time (`0`) using Logistic Regression and a Decision Tree. It reads the CSV file from the project's `data/` folder, so keep the notebook and `data/` folder together.

**Dataset note:** `assignments_synthetic.csv` contains generated sample data for learning. It is not real student data and its evaluation scores are not evidence of real-world performance.

## 1. Install the libraries

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn joblib

## 2. Import packages and set project paths

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

## 3. Load the CSV dataset

The notebook expects `data/assignments_synthetic.csv` relative to the current project directory. If you opened this notebook from the project root, the path below should work.

In [ ]:
csv_path = DATA_DIR / "assignments_synthetic.csv"
df = pd.read_csv(csv_path)
print("Loaded:", csv_path.resolve())
print("Dataset shape:", df.shape)

In [ ]:
display(df.head())
print("Columns:", df.columns.tolist())

## 4. Validate and inspect data

In [ ]:
expected_columns = ["pending_assignments", "study_hours", "previous_late_count",
                    "days_remaining", "assignment_difficulty",
                    "attendance_percentage", "submitted_late"]

In [ ]:
missing_columns = set(expected_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"CSV is missing columns: {sorted(missing_columns)}")

In [ ]:
print("Missing values by column:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.info()
display(df.describe(include="all").T)

## 5. Explore target distribution

In [ ]:
display(df["submitted_late"].value_counts().rename(index={0: "On time (0)", 1: "Late (1)"}))
print("Late proportion:", round(df["submitted_late"].mean(), 3))

In [ ]:
sns.countplot(data=df, x="submitted_late")
plt.title("Assignment Outcome Distribution")
plt.xlabel("0 = On time, 1 = Late")
plt.show()

## 6. Explore feature relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="submitted_late", y="pending_assignments", ax=axes[0])
sns.boxplot(data=df, x="submitted_late", y="study_hours", ax=axes[1])

In [ ]:
axes[0].set_title("Pending Assignments by Outcome")
axes[1].set_title("Study Hours by Outcome")
plt.tight_layout()
plt.show()

In [ ]:
sns.countplot(data=df, x="assignment_difficulty", hue="submitted_late")
plt.title("Difficulty by Assignment Outcome")
plt.show()

## 7. Prepare features and target

In [ ]:
X = df.drop(columns=["submitted_late"]).copy()
y = df["submitted_late"].astype(int).copy()

In [ ]:
numeric_features = ["pending_assignments", "study_hours", "previous_late_count",
                    "days_remaining", "attendance_percentage"]
categorical_features = ["assignment_difficulty"]

## 8. Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [ ]:
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train late rate:", round(y_train.mean(), 3))
print("Test late rate:", round(y_test.mean(), 3))

## 9. Create preprocessing pipeline

Preprocessing is fitted only on training data when the pipeline is trained. This helps prevent test-set leakage. Numeric inputs are imputed and scaled; difficulty is imputed and one-hot encoded.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 10. Train Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

## 11. Train Decision Tree

In [ ]:
tree_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42))
])

In [ ]:
tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)
tree_probabilities = tree_model.predict_proba(X_test)[:, 1]

## 12. Evaluate models

In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {"Model": name, "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),

In [ ]:
            "F1-score": f1_score(y_true, y_pred, zero_division=0)}

In [ ]:
metrics = pd.DataFrame([
    evaluate_model("Logistic Regression", y_test, logistic_predictions),
    evaluate_model("Decision Tree", y_test, tree_predictions)
]).set_index("Model")

In [ ]:
display(metrics.round(3))
metrics.plot(kind="bar", ylim=(0, 1), figsize=(10, 5), rot=0)
plt.title("Model Comparison on Held-Out Test Set")
plt.ylabel("Score")

In [ ]:
plt.tight_layout()
plt.show()

## 13. Classification reports

In [ ]:
print("Logistic Regression report")
print(classification_report(y_test, logistic_predictions,
      target_names=["On time", "Late"], zero_division=0))

In [ ]:
print("Decision Tree report")
print(classification_report(y_test, tree_predictions,
      target_names=["On time", "Late"], zero_division=0))

## 14. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, name, pred in [(axes[0], "Logistic Regression", logistic_predictions),
                       (axes[1], "Decision Tree", tree_predictions)]:

In [ ]:
cm = confusion_matrix(y_test, pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title(name)
ax.set_xlabel("Predicted label")

In [ ]:
ax.set_ylabel("Actual label")
plt.tight_layout()
plt.show()

## 15. Visualize the Decision Tree

In [ ]:
fitted_preprocessor = tree_model.named_steps["preprocessing"]
feature_names = fitted_preprocessor.get_feature_names_out()
fitted_tree = tree_model.named_steps["classifier"]

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(fitted_tree, feature_names=feature_names, class_names=["On time", "Late"],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title("Decision Tree — first three levels")

In [ ]:
plt.show()

## 16. Predict a new assignment

In [ ]:
new_assignment = pd.DataFrame([{
    "pending_assignments": 5, "study_hours": 1.0,
    "previous_late_count": 4, "days_remaining": 2,
    "assignment_difficulty": "Hard", "attendance_percentage": 65.0

In [ ]:
}])

In [ ]:
display(new_assignment)
lr_class = int(logistic_model.predict(new_assignment)[0])
lr_probability = float(logistic_model.predict_proba(new_assignment)[0, 1])

In [ ]:
tree_class = int(tree_model.predict(new_assignment)[0])
tree_probability = float(tree_model.predict_proba(new_assignment)[0, 1])

In [ ]:
print("Logistic Regression:", "Likely late" if lr_class else "Likely on time")
print("Estimated late probability:", round(lr_probability, 3))
print("Decision Tree:", "Likely late" if tree_class else "Likely on time")
print("Estimated late probability:", round(tree_probability, 3))

## 17. Save trained pipelines

In [ ]:
lr_path = MODEL_DIR / "logistic_regression_pipeline.joblib"
tree_path = MODEL_DIR / "decision_tree_pipeline.joblib"
joblib.dump(logistic_model, lr_path)
joblib.dump(tree_model, tree_path)

In [ ]:
print("Saved:", lr_path.resolve())
print("Saved:", tree_path.resolve())

## 18. Reload and verify

In [ ]:
loaded_logistic = joblib.load(lr_path)
loaded_tree = joblib.load(tree_path)
print("Reloaded LR:", loaded_logistic.predict(new_assignment)[0])
print("Reloaded Tree:", loaded_tree.predict(new_assignment)[0])

## 19. Optional batch predictions from CSV

For batch predictions, create a CSV with the six feature columns and no `submitted_late` target column. Uncomment and adapt the example below after you have a file named `data/new_assignments.csv`.

In [ ]:
batch_path = DATA_DIR / "new_assignments.csv"
# batch_df = pd.read_csv(batch_path)
# required = numeric_features + categorical_features
# missing = set(required) - set(batch_df.columns)

In [ ]:
# if missing:
#     raise ValueError(f"Missing columns: {sorted(missing)}")
# batch_df["predicted_late"] = loaded_logistic.predict(batch_df[required])
# batch_df["late_probability"] = loaded_logistic.predict_proba(batch_df[required])[:, 1]

In [ ]:
# batch_df.to_csv(DATA_DIR / "batch_predictions.csv", index=False)
# display(batch_df.head())

## 20. Limitations and responsible use

- The dataset is synthetic and only demonstrates the ML workflow.
- Scores on synthetic data do not prove real-world accuracy.
- Predictions are estimates, not guarantees.
- Do not use this demo to punish students or make high-stakes decisions.
- For real use, collect data with appropriate consent, protect privacy, check for bias, and test on later unseen assignments.

## 21. Conclusion

This notebook loads a CSV, explores data, preprocesses features, trains Logistic Regression and Decision Tree classifiers, evaluates and compares them, predicts a new example, and saves model pipelines. Run cells from top to bottom.